# cancer-recon-apoptosis — RUNG 5: worst-case-safety surfaceome re-audit + the ADDRESSABILITY GAP

The highest-value move (2026-06-01 review): there is **no new-mechanism move** — every modality is already published. The one thing the field's atlases do NOT do is certify safety on the **worst donor / worst cell** instead of the pooled average. So we re-audit surface logic-gate recognition across the **whole OmniPath/SURFY surfaceome**, donor-aware, with **held-out-donor FDR + winner's-curse shrinkage**, and report — per cancer type — the fraction of patients for whom **NO safe gate** clears a coverage bar. That **per-patient addressability gap** is the deliverable. *"Most patients have no clean surface gate"* is the expected, first-class **negative**.

**CPU only — do NOT pick a GPU runtime** (nothing here uses a GPU; it just wastes your free quota). Use a standard CPU runtime; **High-RAM** if offered.

**Memory-safe for free Colab (12 GB).** The full surfaceome is ~5,000 genes — too big to hold × the 10 M-cell brain. So the run is **two-pass**: it screens the whole surfaceome on the small tumour set, then carries only the **tumour-expressed** genes at depth, and **caps cells at the query level** so no tissue ever materialises in full. Each tissue **caches to Drive** (Cell 2), so a disconnect/sleep just **resumes** — Run all again.

In [ ]:
#@title Cell 1 — clone / pull repo
import os
from pathlib import Path
REPO = Path('/content/cancer-recon-apoptosis')
if REPO.exists():
    !cd {REPO} && rm -f runs/logs/*.log && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recon-apoptosis.git {REPO}
os.chdir(REPO)
!git log --oneline -1
print('[CELL 1] ✓')

In [ ]:
#@title Cell 2 — mount Google Drive for a RESUMABLE cache (recommended) + start run log
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    cache_dir = '/content/drive/MyDrive/cancer-recon'
    os.makedirs(cache_dir, exist_ok=True)
    # scripts/25 -> *.r5.tumour.npz, *.r5.normal.npz, and per-tissue tiles in r5_normal_tiles/ (resumable).
    os.environ['LOGICGATE_CACHE'] = f'{cache_dir}/rung5_cache.npz'
    print('[CELL 2] Drive mounted — fetch cached to', os.environ['LOGICGATE_CACHE'])
    print('         If it disconnects mid-fetch, just Run-all again: per-tissue tiles resume (no full re-fetch).')
except Exception as e:
    os.environ['LOGICGATE_CACHE'] = '/content/cancer-recon-apoptosis/data/logicgate/rung5_cache.npz'
    print('[CELL 2] Drive NOT mounted (', type(e).__name__, ') — caching to /content (LOST on disconnect).')
from scripts.runlog import new_log
RUNLOG = new_log('rung5_addressability', repo_dir='.')
print('[CELL 2] ✓')

In [ ]:
#@title Cell 3 — VALIDATE the rigor layer + the full downstream on synthetic ground truth (CPU, no data)
# Proves BEFORE any Census fetch: (a) family-max FDR + winner's-curse shrinkage catch the look-elsewhere /
# single-donor reward-hacks (scripts/23), and (b) the whole donor-aware pipeline runs end-to-end and
# computes the addressability gap (scripts/25 selftest). If either fails, do NOT trust the real run.
import sys
from scripts.runlog import run_logged
rc1 = run_logged([sys.executable, '-u', 'scripts/23_heldout_validate.py'], RUNLOG)
rc2 = run_logged([sys.executable, '-u', 'scripts/25_logicgate_data_rung5.py', 'selftest'], RUNLOG)
print(f'[CELL 3] scripts/23 exit={rc1}  scripts/25 selftest exit={rc2}',
      '✓ both validated' if rc1 == 0 and rc2 == 0 else '✗ validation FAILED — stop here')
from IPython.display import Image, display
import os
for p in ['runs/rung5_logicgate/rung5_heldout_validation.png', 'runs/rung5_logicgate/rung5_selftest.png']:
    if os.path.exists(p):
        display(Image(p))

In [ ]:
#@title Cell 4 — install CELLxGENE Census + OmniPath (only needed for the real run)
import sys, importlib.util
from scripts.runlog import run_logged
for pkg, pip_name in [('cellxgene_census', 'cellxgene-census'), ('scanpy', 'scanpy'), ('omnipath', 'omnipath')]:
    if importlib.util.find_spec(pkg) is None:
        run_logged([sys.executable, '-m', 'pip', 'install', '-q', pip_name], RUNLOG, label=f'pip install {pip_name}')
print('[CELL 4] ✓ (if Colab asks to RESTART runtime, do it, then Run all again — the cache makes that cheap)')

In [ ]:
#@title Cell 5 — REAL run: donor-aware addressability gap (memory-safe two-pass; resumes from cache)
#@markdown **Free-tier defaults** below finish in ~1–2 h on free Colab (fetch + a coarse-but-real FDR). For the
#@markdown exhaustive map, raise MAX_FAMILY/N_PERM/N_BOOT on a High-RAM/Pro runtime (or after the vectorised scorer lands).
K_ACTIVATORS = 60   #@param {type:"integer"}
COV_FLOOR = 0.05    #@param {type:"number"}
GENE_FLOOR = 0.02   #@param {type:"number"}
MAX_FAMILY = 1200   #@param {type:"integer"}
N_PERM = 30         #@param {type:"integer"}
N_BOOT = 30         #@param {type:"integer"}
import os, sys
os.environ.update(R5_K_ACTIVATORS=str(K_ACTIVATORS), R5_COV_FLOOR=str(COV_FLOOR), R5_GENE_FLOOR=str(GENE_FLOOR),
                  R5_MAX_FAMILY=str(MAX_FAMILY), R5_N_PERM=str(N_PERM), R5_N_BOOT=str(N_BOOT))
rc = run_logged([sys.executable, '-u', 'scripts/25_logicgate_data_rung5.py'], RUNLOG)
print(f'[CELL 5] exit={rc}')
import json
p = 'runs/rung5_logicgate/rung5_addressability.json'
if os.path.exists(p):
    r = json.load(open(p))
    a = r.get('addressability', {})
    print('full surfaceome / screened-expressed:', r.get('surfaceome_full'), '->', r.get('surfaceome_screened_expressed'))
    print('cells/donors/genes :', r.get('n_cells'), '/', r.get('n_donors'), '/', r.get('n_genes'))
    print('family size        :', r.get('family_size'), '| SAFE:', r.get('n_safe'),
          '| survive FDR+shrinkage:', r.get('n_survivors'))
    print('ADDRESSABILITY GAP :', a.get('addressability_gap_overall'), 'overall')
    print('  by cancer type   :', a.get('addressability_gap_by_cancer_type'))
    print('survivors (top)    :', r.get('survivors'))
from IPython.display import Image, display
if os.path.exists('runs/rung5_logicgate/rung5_real.png'):
    display(Image('runs/rung5_logicgate/rung5_real.png'))

In [ ]:
#@title Cell 6 — finalize + download run log AND all figures / result files
import os, glob
from scripts.runlog import finalize
finalize(RUNLOG)   # downloads the commit-stamped log to your Downloads folder
EXTRA = (glob.glob('runs/rung5_logicgate/*.png') + glob.glob('runs/rung5_logicgate/*.json')
         + glob.glob('data/logicgate/surfaceome_genes.txt'))
try:
    from google.colab import files
    for p in sorted(set(EXTRA)):
        if os.path.exists(p):
            print('[download]', p); files.download(p)
except ImportError:
    print('(not in Colab — download skipped)')
except Exception as e:
    print('(download skipped:', type(e).__name__, e, ')')

## What this rung establishes

- **The harness is validated before the data** (Cell 3): family-max FDR + winner's-curse shrinkage catch the look-elsewhere and single-lucky-donor reward-hacks; the full donor-aware pipeline computes the addressability gap end-to-end on synthetic ground truth.
- **The real run** is worst-DONOR safe (never pooled), fail-closed on vital organs, dropout-guarded, no-multiply; coverage must beat the **family-max decoy FDR** and survive **winner's-curse shrinkage** on held-out donors.
- **The deliverable is the per-patient ADDRESSABILITY GAP** — the fraction of patients (by cancer type) with no safe gate. *"Most patients have no clean surface gate"* is the expected, first-class **negative**, and the honest, quantified reason the field resorts to HLA-LOH NOT-gates.

**Free-tier note:** the defaults give a **coarse-but-real** first pass (smaller search, fewer permutations) — clearly labeled, not faked. The exhaustive map wants more compute; raise the knobs on a High-RAM/Pro runtime.

**Ceiling:** transcript-only (mRNA ≠ surface protein; CITE-seq confirms co-positivity); co-localisation ≠ a firing circuit (agonism is wet-lab); NOT first at combinatorial search — the contribution is the worst-case-safety honesty harness + the addressability gap. A surviving gate is the best NEXT wet-lab experiment, not a cure.